# 05 · How far from shore? Siting and layout together — *work in progress*

Moving a farm further offshore gives stronger, steadier wind but a longer export cable back to land.
This notebook adds distance to shore as a design variable alongside the turbine positions and
optimises three objectives at once:

1. maximise AEP
2. minimise inter-array cable (MST, as in notebooks 03–04)
3. minimise export cable (landfall to a substation at the farm centroid)

The export cable is kept as a separate objective rather than added to the array cable, because the
two are different products with different copper content per km.

> **Status.** How wind speed changes with distance from the coast is currently a placeholder
> (see the assumptions cell below), and the copper-per-km figures are not filled in yet.
> The optimisation machinery works; the numbers it produces should not be read as results until
> both are replaced with sourced data.

In [ ]:
%pip install -q py_wake==2.6.20 topfarm==2.6.2 pymoo==0.6.2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree

from py_wake.site import XRSite
from py_wake.site.xrsite import GlobalWindAtlasSite
from py_wake.examples.data.dtu10mw import DTU10MW
from py_wake.literature.gaussian_models import Niayifar_PorteAgel_2016
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

FIG = Path("figures"); FIG.mkdir(exist_ok=True)

In [ ]:
DATA = Path("data/dogger_bank_gwa.nc")

def load_dogger_bank_site():
    """Dogger Bank wind climate from the Global Wind Atlas (54.75 N, 1.90 E, 119 m, open sea).
    Downloaded once and cached in data/ so later runs don't depend on the GWA API."""
    if DATA.exists():
        return XRSite(xr.load_dataset(DATA))
    print("Downloading Dogger Bank wind climate from the Global Wind Atlas...")
    site = GlobalWindAtlasSite(lat=54.75, long=1.90, height=119, roughness=0.0002)
    DATA.parent.mkdir(exist_ok=True)
    site.ds.to_netcdf(DATA)
    return site

## Assumptions

- The Global Wind Atlas point used in notebook 04 is taken to lie about **150 km** from the coast,
  and its wind climate is used as the reference (relative wind speed = 1.0 there).
- Closer to shore, wind speed is scaled down with a saturating recovery curve,
  `1 − a·exp(−d / L)`, with **a = 0.15 and L = 20 km. These two values are placeholders**, not fitted
  to data. The next step is to replace them with a Global Wind Atlas transect running out from the coast.
- The wind rose shape (direction frequencies) is assumed not to change with distance.

In [ ]:
wt = DTU10MW()
TI = 0.06
n_wt = 8
MIN_SPACING = 4 * wt.diameter()     # 713 m
LEASE_X, LEASE_Y = 3000, 2000
D_REF_KM, D_MIN_KM, D_MAX_KM = 150, 10, 150
POP_SIZE, N_GEN, SEED = 80, 100, 1

A_REC, L_REC_KM = 0.15, 20.0        # PLACEHOLDERS - see assumptions above

d_grid_km = np.arange(0, D_MAX_KM + 5, 0.25)
recovery = lambda d: 1 - A_REC * np.exp(-d / L_REC_KM)
U_rel = recovery(d_grid_km) / recovery(D_REF_KM)

# Dogger Bank wind climate, with a speed-up factor that varies along x (= distance offshore)
ds = load_dogger_bank_site().ds.copy().drop_vars("h", errors="ignore")
y_grid = np.array([-5e3, 10e3])
ds = ds.assign_coords(x=d_grid_km * 1e3, y=y_grid)
ds["Speedup"] = (("x", "y"), np.repeat(U_rel[:, None], len(y_grid), axis=1))
macro_site = XRSite(ds, interp_method="linear")
wf_macro = Niayifar_PorteAgel_2016(macro_site, wt)

## Sanity check before optimising

In [ ]:
x0 = np.linspace(0, 2400, 4).repeat(2)
y0 = np.tile(np.linspace(0, 1600, 2), 4)
aep_ref = wf_macro(x0 + D_REF_KM * 1e3, y0, TI=TI).aep().sum().values
aep_near = wf_macro(x0 + D_MIN_KM * 1e3, y0, TI=TI).aep().sum().values
u_ratio = np.interp(D_MIN_KM, d_grid_km, U_rel)

print(f"AEP at {D_REF_KM} km: {aep_ref:.1f} GWh | at {D_MIN_KM} km: {aep_near:.1f} GWh")
print(f"Wind speed ratio {D_MIN_KM} km / {D_REF_KM} km: {u_ratio:.3f}")
print(f"AEP ratio: {aep_near / aep_ref:.3f}  (cube of wind ratio would give {u_ratio ** 3:.3f}; "
      "AEP falls less than that because turbines cap out at rated power)")

In [ ]:
def array_cable_km(x, y):
    return minimum_spanning_tree(squareform(pdist(np.column_stack([x, y])))).sum() / 1000


class SitingAndLayout(ElementwiseProblem):
    def __init__(self):
        xl = [0] * (2 * n_wt) + [D_MIN_KM]
        xu = [LEASE_X] * n_wt + [LEASE_Y] * n_wt + [D_MAX_KM]
        super().__init__(n_var=len(xl), n_obj=3, n_ieq_constr=1, xl=xl, xu=xu)

    def _evaluate(self, v, out, *args, **kwargs):
        x, y, d_shore = v[:n_wt], v[n_wt:2 * n_wt], v[-1]
        x_abs = x + d_shore * 1e3                 # lease box starts d_shore km offshore
        aep = wf_macro(x_abs, y, TI=TI).aep().sum().values
        export_km = d_shore + x.mean() / 1000     # landfall -> substation at centroid
        out["F"] = [-aep, array_cable_km(x, y), export_km]
        out["G"] = [MIN_SPACING - pdist(np.column_stack([x, y])).min()]


def initial_population(problem, seed_row):
    # Random layouts, with the starting grid as the first member so the search
    # always has one feasible, sensible layout to build from.
    rng = np.random.default_rng(SEED)
    X0 = rng.uniform(problem.xl, problem.xu, size=(POP_SIZE, problem.n_var))
    X0[0] = seed_row
    return X0


problem = SitingAndLayout()
X0 = initial_population(problem, np.concatenate([x0, y0, [D_REF_KM]]))
result = minimize(problem, NSGA2(pop_size=POP_SIZE, sampling=X0), ("n_gen", N_GEN), seed=SEED, verbose=False)

front = pd.DataFrame({
    "Distance_to_shore_km": result.X[:, -1],
    "AEP_GWh": -result.F[:, 0],
    "Array_cable_km": result.F[:, 1],
    "Export_cable_km": result.F[:, 2],
}).sort_values("Distance_to_shore_km").reset_index(drop=True)
print(front.round(2).to_string())

## Copper (to be filled in)

Copper is calculated after the optimisation rather than inside it, so the Pareto front does not
depend on the copper intensity values. Set the two constants below (tonnes of copper per km for
array and export cable, from a cited source) to add a copper column.

In [ ]:
CU_ARRAY_T_PER_KM = None    # TODO: add value and source
CU_EXPORT_T_PER_KM = None   # TODO: add value and source

if CU_ARRAY_T_PER_KM is not None and CU_EXPORT_T_PER_KM is not None:
    front["Copper_t"] = (front.Array_cable_km * CU_ARRAY_T_PER_KM
                         + front.Export_cable_km * CU_EXPORT_T_PER_KM)
    print(front.round(1).to_string())
else:
    print("Copper intensities not set yet - skipping copper column.")

In [ ]:
plt.figure(figsize=(9, 5))
sc = plt.scatter(front.Export_cable_km, front.AEP_GWh, c=front.Array_cable_km, cmap="viridis")
plt.colorbar(sc, label="Inter-array cable, MST (km)")
plt.xlabel("Export cable (km)"); plt.ylabel("AEP (GWh)")
plt.title("Siting + layout trade-off (placeholder wind-speed gradient)")
plt.grid(True, alpha=0.4)
plt.savefig(FIG / "05_siting_front.png", dpi=150, bbox_inches="tight")
plt.show()